In [ ]:
import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup

#### Fetch Play Records

In [ ]:
scraper = cloudscraper.create_scraper(
    disableCloudflareV1=True,
    browser={
        "browser": "chrome",
        "platform": "windows",
        "mobile": False,
    },
    delay=10,
)

In [ ]:
season = "2021-2022"
pages, links = [], []

response = scraper.get(
    "https://www.forebet.com/en/football-tips-and-predictions-for-japan/"
    f"j1-league/results/{season}"
)
soup = BeautifulSoup(response.text, "html.parser")

for page in soup.find_all("a", class_="pagenav", href=True):
    if page.text.isnumeric():
        pages.append(page)

links += soup.find_all("a", class_="stat_link", href=True)

for page in pages:
    response = scraper.get("https://www.forebet.com" + page["href"])
    soup = BeautifulSoup(response.text, "html.parser")

    links += soup.find_all("a", class_="stat_link", href=True)

#### Download Play History

In [ ]:
data = []

for link in links:
    file_name = link["href"].split("/")[-1]
    pth = f"../data/cache/{season}/{file_name}.html"

    response = scraper.get("https://www.forebet.com" + link["href"])
    html = response.text
    pth.write_text(html, encoding="utf-8")

    soup = BeautifulSoup(html, "html.parser")

    date = soup.find("div", class_="date_bah")
    home = soup.find("span", class_="homeTeam")
    away = soup.find("span", class_="awayTeam")
    full_goal = soup.find("b", class_="l_scr")
    half_goal = soup.find("span", class_="ht_scr")
    handicap = soup.find("span", class_="forepr ashc")
    source = file_name

    data.append([
        date.text if date else None,
        home.text if home else None,
        away.text if away else None,
        full_goal.text if full_goal else None,
        half_goal.text if half_goal else None,
        handicap.text if handicap else None,
        source,
    ])

In [ ]:
col = ["date", "home", "away", "full_goal", "half_goal", "handicap", "source"]
df = pd.DataFrame(data, columns=col)

df.to_parquet(f"../data/raw/{season}.parquet", index=False)